In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
import json

# ==========================================
# Event Hub Connection
# ==========================================

connectionString = (
    "Endpoint=sb://eh-social-media.servicebus.windows.net/;"
    "SharedAccessKeyName=sudheer;"
    "SharedAccessKey=1IhrynvVGzNbhQLyvqMUxuT7QdTRHRh57+AEhJK7+eQ=;"
    "EntityPath=valid-tweets-hub"
)

ehConf = {}

ehConf["eventhubs.connectionString"] = (
    sc._jvm.org.apache.spark.eventhubs.EventHubsUtils.encrypt(connectionString)
)

ehConf["eventhubs.consumerGroup"] = "$Default"

startingEventPosition = {
    "offset": "-1",
    "seqNo": -1,
    "enqueuedTime": None,
    "isInclusive": True
}

ehConf["eventhubs.startingPosition"] = json.dumps(startingEventPosition)
ehConf["eventhubs.maxEventsPerTrigger"] = "5000"

# ==========================================
# Read Stream
# ==========================================

rawDF = (
    spark.readStream
         .format("eventhubs")
         .options(**ehConf)
         .load()
)

eventDF = rawDF.selectExpr(
    "CAST(body AS STRING) AS message"
)

# ==========================================
# Schema
# ==========================================

schema = StructType([
    StructField("tweet_id", StringType(), True),
    StructField("topic_category", StringType(), True),
    StructField("tweet_text", StringType(), True),
    StructField("tweet_timestamp", StringType(), True),
    StructField("impressions", DoubleType(), True),
    StructField("likes", DoubleType(), True),
    StructField("retweets", DoubleType(), True),
    StructField("replies", DoubleType(), True),
    StructField("engagement_count", DoubleType(), True),
    StructField("sentiment_score", DoubleType(), True)
])

# ==========================================
# Parse JSON
# ==========================================

parsedDF = (
    eventDF
        .select(from_json(col("message"), schema).alias("data"))
        .select("data.*")
)

# ==========================================
# Error Records
# ==========================================

errorDF = parsedDF.filter(col("tweet_id").isNull())

errorQuery = (
    errorDF.writeStream
        .trigger(availableNow=True)
        .format("delta")
        .outputMode("append")
        .option(
            "checkpointLocation",
            "abfss://raw@socialmedia1.dfs.core.windows.net/checkpoints/error_bronze_valid_tweets"
        )
        .toTable("socialmedia_databricks.error.error_bronze_valid_tweets")
)

# ==========================================
# Bronze Data
# ==========================================

bronzeDF = (
    parsedDF
    .filter(col("tweet_id").isNotNull())

    .withColumn(
        "tweet_timestamp",
        to_timestamp(
            col("tweet_timestamp"),
            "dd-MM-yyyy HH:mm"
        )
    )

    .withColumn("bronze_load_time", current_timestamp())
    .withColumn("pipeline_name", lit("Bronze_Valid_Tweets"))
    .withColumn("source_system", lit("Azure Event Hub"))
    .withColumn("ingestion_date", current_date())

    .withWatermark("tweet_timestamp", "10 minutes")
)

# ==========================================
# Write Bronze Table
# ==========================================

bronzeQuery = (
    bronzeDF.writeStream
        .trigger(availableNow=True)
        .format("delta")
        .outputMode("append")
        .option(
            "checkpointLocation",
            "abfss://raw@socialmedia1.dfs.core.windows.net/checkpoints/bronze_valid_tweets"
        )
        .option("mergeSchema", "true")
        .toTable("socialmedia_databricks.bronze.bronze_valid_tweets")
)

bronzeQuery.awaitTermination()
errorQuery.awaitTermination()

print("Bronze Valid Tweets Loaded Successfully")

Bronze Valid Tweets Loaded Successfully


In [0]:
%sql
select * from socialmedia_databricks.bronze.bronze_valid_tweets;

tweet_id,topic_category,tweet_text,tweet_timestamp,impressions,likes,retweets,replies,engagement_count,sentiment_score,bronze_load_time,pipeline_name,source_system,ingestion_date
"""NaN""",Sports,Spark > Hadoop ???,2025-01-15T16:33:00Z,5714.0,3553.0,NaN,82.0,948.0,0.708690574,2026-07-09T07:12:14.051Z,Bronze_Valid_Tweets,Azure Event Hub,2026-07-09
T36356,Sports,Testing!!!@@@###,null,11387.0,4616.0,624.0,2901.0,1203.0,NaN,2026-07-09T07:12:14.051Z,Bronze_Valid_Tweets,Azure Event Hub,2026-07-09
T7423,Cloud,Worst service ever :(,null,4661.0,1921.0,155.0,383.0,476.0,0.851583872,2026-07-09T07:12:14.051Z,Bronze_Valid_Tweets,Azure Event Hub,2026-07-09
T19259,Sports,Streaming $$$ working,2025-01-04T12:12:00Z,13416.0,1101.0,876.0,1506.0,191.0,0.550269347,2026-07-09T07:12:14.051Z,Bronze_Valid_Tweets,Azure Event Hub,2026-07-09
T32081,Sports,Data pipeline broke @ midnight!!!,2025-01-17T23:44:00Z,1056.0,1259.0,76.0,2322.0,248.0,0.083282515,2026-07-09T07:12:14.051Z,Bronze_Valid_Tweets,Azure Event Hub,2026-07-09
T20773,Sports,Streaming $$$ working,2025-01-07T14:05:00Z,5328.0,4917.0,916.0,2023.0,1146.0,0.102938871,2026-07-09T07:12:14.051Z,Bronze_Valid_Tweets,Azure Event Hub,2026-07-09
"""NaN""",Finance,Love this!!! 😊,2025-01-15T10:27:00Z,14153.0,434.0,603.0,1359.0,1218.0,0.104350305,2026-07-09T07:12:14.051Z,Bronze_Valid_Tweets,Azure Event Hub,2026-07-09
T31184,Sports,Streaming $$$ working,2025-01-22T01:06:00Z,1763.0,2403.0,21.0,151.0,1357.0,0.842351462,2026-07-09T07:12:14.051Z,Bronze_Valid_Tweets,Azure Event Hub,2026-07-09
T24636,Finance,Worst service ever :(,2025-01-12T01:22:00Z,441.0,437.0,652.0,1043.0,583.0,0.462374657,2026-07-09T07:12:14.051Z,Bronze_Valid_Tweets,Azure Event Hub,2026-07-09
T10424,Finance,Love this!!! 😊,2025-01-15T19:48:00Z,2736.0,1959.0,796.0,716.0,NaN,0.212739958,2026-07-09T07:12:14.051Z,Bronze_Valid_Tweets,Azure Event Hub,2026-07-09
